[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bulentsoykan/ABM-particle-filter-notebooks/blob/main/notebook_01_abm_uncertainty.ipynb)

# Module 1: Why Agent-Based Models Drift

**Learning objectives**
1. Understand what an Agent-Based Model (ABM) is and what makes it stochastic
2. See how small random choices compound into large state uncertainty over time
3. Understand *why* real-time data assimilation is necessary



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.alpha': 0.5,  'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'axes.grid': True,
    'font.size': 11,
})
os.makedirs('figures', exist_ok=True)
print("Libraries loaded.")

## The Model: A Simple Pedestrian Corridor

We use a minimal version of **StationSim** from the paper (Malleson et al. 2020).
Agents enter from the left and walk to the right exit through a 100 m × 50 m corridor.

Each agent has:
- A **random maximum speed** (drawn from a Gaussian, mean = 1.0 m/step, std = 0.4)
- One collision-avoidance rule: when blocked by a slower agent ahead,
  randomly dodge **LEFT or RIGHT** (50/50 coin flip)

That single binary random event — dodge left *or* right — is the **only stochasticity**.
Everything else is deterministic. Yet as we'll see, it causes enormous divergence.

```
   ┌─────────────────────────────────────────────────┐
   │  → → →  (agents cross)                          │
   │     ↗ (fast catches slow, random dodge)         │
   └─────────────────────────────────────────────────┘
   [ENTER x=0]                             [EXIT x=100]
```


In [ ]:
# ── Minimal StationSim-style corridor ABM ──────────────────────────────────────
#    Based on Malleson et al. (2020), simplified for teaching.
#    Paper repo: github.com/Urban-Analytics/dust

class Agent:
    '''A pedestrian moving left to right through a corridor.

    The ONLY source of randomness: when a faster agent catches up to a slower one,
    it randomly dodges LEFT or RIGHT (50/50 coin flip).  Everything else is
    deterministic.  That one binary choice is all it takes to cause divergence.
    '''
    def __init__(self, agent_id, env_width=100, env_height=50, rng=None):
        rng = rng or np.random
        self.id         = agent_id
        self.x          = rng.uniform(0, 3)
        self.y          = rng.uniform(5, env_height - 5)
        self.max_speed  = max(0.3, rng.normal(1.0, 0.4))
        self.env_width  = env_width
        self.env_height = env_height
        self.active     = True
        self.traj_x     = [self.x]
        self.traj_y     = [self.y]

    def step(self, all_agents, separation=4.5, rng=None):
        rng = rng or np.random
        if not self.active:
            self.traj_x.append(self.x)
            self.traj_y.append(self.y)
            return
        speed, dy = self.max_speed, 0.0
        for other in all_agents:
            if other.id == self.id or not other.active:
                continue
            dx   = other.x - self.x
            dist = np.hypot(dx, other.y - self.y)
            if 0 < dx < separation * 2 and dist < separation:
                speed *= 0.6
                dy = 1.0 if rng.random() < 0.5 else -1.0   # THE random choice
                break
        self.x = min(self.x + speed, self.env_width)
        self.y = np.clip(self.y + dy, 2, self.env_height - 2)
        if self.x >= self.env_width:
            self.active = False
        self.traj_x.append(self.x)
        self.traj_y.append(self.y)


class CorridorABM:
    '''Corridor where N agents walk left-to-right.

    Simplified StationSim from Malleson et al. (2020).
    State vector: [x0,y0, x1,y1, ..., x_{N-1},y_{N-1}]  — dimension 2*N_agents.
    '''
    def __init__(self, n_agents=10, env_width=100, env_height=50, seed=None):
        rng             = np.random.RandomState(seed)
        self.n_agents   = n_agents
        self.env_width  = env_width
        self.env_height = env_height
        self.rng        = rng
        self.agents     = [Agent(i, env_width, env_height, rng)
                           for i in range(n_agents)]
        self.pos_history = [self.get_positions()]
        self.step_count  = 0
        self.collisions  = 0

    def get_positions(self):
        return np.array([[a.x, a.y] for a in self.agents])

    def get_state_vector(self):
        '''Flat [x0,y0, x1,y1,...] — used by the particle filter.'''
        return self.get_positions().flatten()

    def step(self):
        before = [a.active for a in self.agents]
        for agent in self.agents:
            agent.step(self.agents, rng=self.rng)
        # count collision events (agents that slowed down)
        for a in self.agents:
            for b in self.agents:
                if a.id >= b.id or not a.active or not b.active:
                    continue
                if np.hypot(a.x - b.x, a.y - b.y) < 4.5:
                    self.collisions += 1
        self.pos_history.append(self.get_positions())
        self.step_count += 1

    def run(self, n_steps):
        for _ in range(n_steps):
            self.step()
        return self

print("CorridorABM defined.  State dimension = 2 x N_agents.")


## Running the Identical Twin Experiment

One model is the **pseudo-truth** (the simulated "real world").
`N_ENS` copies run with different random seeds — same parameters, different outcomes.

If the model were deterministic, all runs would be identical. Watch what actually happens.


In [ ]:
N_AGENTS, N_STEPS, N_ENS = 12, 80, 20

# Pseudo-truth: the "real world" whose state we want to track
truth    = CorridorABM(n_agents=N_AGENTS, seed=0).run(N_STEPS)

# Ensemble: independent simulations representing our uncertainty
ensemble = [CorridorABM(n_agents=N_AGENTS, seed=100 + i).run(N_STEPS)
            for i in range(N_ENS)]

# Compute ensemble spread: mean std across agents and (x,y) at each time step
# all_pos shape: (N_ENS, N_STEPS+1, N_AGENTS, 2)
all_pos = np.array([[m.pos_history[t] for t in range(N_STEPS + 1)]
                    for m in ensemble])
spread  = np.mean(np.std(all_pos, axis=0), axis=(1, 2))

print(f"Ensemble spread (mean std across all agents):")
print(f"  Step   0 : {spread[0]:.3f} m  (initial scatter from random start positions)")
print(f"  Step  {N_STEPS//2:2d} : {spread[N_STEPS//2]:.3f} m")
print(f"  Step  {N_STEPS:2d} : {spread[-1]:.3f} m")
print(f"\nUncertainty grew {spread[-1]/max(spread[1],0.01):.0f}x in {N_STEPS} steps!")


In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = GridSpec(2, 3, fig, hspace=0.4, wspace=0.32)
BLUES = plt.cm.Blues(np.linspace(0.25, 0.65, N_ENS))
steps = np.arange(N_STEPS + 1)

# ── A: 2D trajectory spaghetti ────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, :2])
for k, m in enumerate(ensemble):
    for ag in m.agents:
        ax_a.plot(ag.traj_x, ag.traj_y, color=BLUES[k], alpha=0.22, lw=0.7)
for ag in truth.agents:
    ax_a.plot(ag.traj_x, ag.traj_y, color='#f9826c', lw=2.0, zorder=5)
ax_a.axvline(0,   color='#3fb950', lw=2, ls='-')
ax_a.axvline(100, color='#f85149', lw=2, ls='-')
ax_a.set(xlim=(-2, 105), ylim=(0, 50),
         xlabel='X position (m)', ylabel='Y position (m)',
         title='A   Trajectory Spaghetti — Ensemble (blue) vs Pseudo-Truth (orange)')
ax_a.legend(handles=[
    mpatches.Patch(fc='#1f6feb', alpha=0.5, label=f'{N_ENS} ensemble runs'),
    mpatches.Patch(fc='#f9826c',             label='Pseudo-truth'),
    plt.Line2D([0],[0], color='#3fb950', lw=2, label='Entrance'),
    plt.Line2D([0],[0], color='#f85149', lw=2, label='Exit'),
], fontsize=9, loc='upper left')

# ── B: Agent-0 X-position divergence ─────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 2])
ens_x0 = np.array([m.agents[0].traj_x[:N_STEPS+1] for m in ensemble])
mu, sg = np.mean(ens_x0, axis=0), np.std(ens_x0, axis=0)
for k, row in enumerate(ens_x0):
    ax_b.plot(steps, row, color=BLUES[k], alpha=0.3, lw=0.7)
ax_b.fill_between(steps, mu - 2*sg, mu + 2*sg, alpha=0.2, color='cyan',
                  label='+-2 sigma band')
ax_b.plot(steps, mu, color='cyan', lw=2, label='Ensemble mean')
ax_b.plot(steps, truth.agents[0].traj_x[:N_STEPS+1],
          color='#f9826c', lw=2.5, label='Pseudo-truth')
ax_b.set(xlabel='Time step', ylabel='X of agent 0 (m)',
         title='B   Single-Agent Divergence')
ax_b.legend(fontsize=9)

# ── C: Ensemble spread over time ──────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, :2])
ax_c.plot(steps, spread, color='#f0883e', lw=2.5, label='Mean ensemble spread')
ax_c.fill_between(steps, 0, spread, alpha=0.2, color='#f0883e')
ax_c.axhline(spread[1], color='#8b949e', ls='--', lw=1,
             label=f'Initial spread ~ {spread[1]:.2f} m')
ax_c.set(xlabel='Time step', ylabel='Mean std across agents (m)',
         title='C   Ensemble Spread — The "Uncertainty Fan"')
ax_c.legend(fontsize=9)

# ── D: Explanation annotation ─────────────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 2])
ax_d.axis('off')
msg = ("THE RANDOM FORK\n"
       "\u2500" * 28 + "\n\n"
       "Each near-collision forces\n"
       "a binary coin flip:\n\n"
       "  dodge LEFT  (p = 0.5)\n"
       "  dodge RIGHT (p = 0.5)\n\n"
       f"{N_AGENTS} agents x {N_STEPS} steps\n"
       "= hundreds of forks\n\n"
       "Two identical-start runs\n"
       "diverge completely.\n\n"
       "We CANNOT know the true\n"
       "state without data.\n\n"
       "=> Module 3 solves this\n"
       "   with a Particle Filter.")
ax_d.text(0.05, 0.97, msg, transform=ax_d.transAxes, va='top', fontsize=9.5,
          fontfamily='monospace',
          bbox=dict(boxstyle='round', fc='#1f6feb22', ec='#1f6feb', alpha=0.9))

plt.suptitle('Module 1 - Agent-Based Model Uncertainty', fontsize=15, fontweight='bold')
plt.savefig('fig_01_abm_uncertainty.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_01_abm_uncertainty.png")


## Key Takeaways

| Concept | What we observed |
|---------|-----------------|
| **Agent heterogeneity** | Each agent has a different max speed — different crowding patterns |
| **Emergent stochasticity** | Randomness arises from *interactions*, not baked into individual agents |
| **State dimension** | 12 agents → **24-dimensional state space** (x, y per agent) |
| **Uncertainty fans out** | Small per-step randomness compounds into large state uncertainty |
| **The fundamental problem** | Even a perfect model cannot know the true state without data |

**Next: Module 2** introduces the Particle Filter — the algorithm that uses observations
to pull ensemble members back towards the truth at each time step.
